# TN2 — Tầm nhìn DS-TCN 192 kênh, 4 fold

## Vì sao chạy thang này

### Kết quả ĐÃ CÓ của bản 64 kênh — notebook này KHÔNG chạy lại

Vòng sàng lọc một fold trên 64 kênh cho kết quả đơn điệu — ba cấu hình
notebook này chạy:

| kernel | tầm nhìn | điểm `val_KL` | train_mse | train_pearson |
|---:|---:|---:|---:|---:|
| 5 | 121 | **0,8458** | 0,02127 | 0,5851 |
| 7 | 181 | 0,8176 | 0,02059 | 0,5921 |
| 9 | 241 | 0,8041 | 0,01939 | 0,6041 |

Tương quan tầm nhìn với điểm: **−0,983**. Tầm nhìn càng dài, chọn kênh càng tệ.

Hai cấu hình tầm nhìn dài hơn cũng đã chạy và **tệ hơn cả ba dòng trên**:
tầm nhìn 301 được 0,7940 và tầm nhìn 361 được 0,7759. Bỏ khỏi vòng này.

Và cùng lúc điểm tụt thì model **dự báo giỏi lên** — `train_mse` giảm 13%,
`train_pearson` tăng. Model càng giỏi dự báo càng chọn kênh dở.

Giả thuyết: tiêu chí chọn kênh là "ứng viên nào tự dự báo được chính nó tốt
nhất". Model tầm nhìn ngắn chỉ đoán giỏi sóng **thật sự tuần hoàn**; model tầm
nhìn dài đoán giỏi **mọi sóng trơn**, kể cả kênh nhiễu có cấu trúc. Tầm nhìn
ngắn hoạt động như bộ lọc — dở đúng chỗ cần dở.

**Notebook này xem xu hướng đó có lặp lại ở 192 kênh không.** Nếu có, giả thuyết
mạnh hơn hẳn: hai bề rộng kênh khác nhau 8 lần mà cùng một xu hướng.

## Chạy gì

Ba cấu hình, **đủ bốn fold**, một seed. Bỏ hai tầm nhìn dài nhất vì bản 64 kênh
đã cho thấy chúng tệ nhất.

| kernel | tầm nhìn | tham số | phủ cửa sổ 200 |
|---:|---:|---:|---:|
| 5 | 121 | 310.873 | 60% |
| 7 | 181 | 313.945 | 90% |
| 9 | 241 | 317.017 | 100% |

Điểm nối đầu thang, đã có: **kernel 3, tầm nhìn 61, 307.801 tham số**.

## Thời gian

Tuỳ đã chạy vòng sàng lọc `TN2_ReceptiveField_DS_TCN_c192` chưa:

    đã chạy      fold val_KL được bỏ qua  ->  9 fold  ->  khoảng 1,8 giờ
    chưa chạy    làm đủ 12 fold           ->            khoảng 2,4 giờ

Ô khôi phục ở mục 1 tự lo phần này — nó kéo mọi kết quả `tn2_rf` đã có từ Drive
về, và `run_cv.py` bỏ qua fold nào đã xong.

## Chưa kết luận được sau vòng này

Một seed. `seed_std` của tám cấu hình TN1 trải từ 0,0007 tới 0,0108, mỗi kiến
trúc một khác — không mượn của nhau được.

Vòng này trả lời câu hẹp hơn: **xu hướng ở 64 kênh có lặp lại ở 192 kênh
không**, và **có giữ nguyên trên cả bốn fold không**. Nếu `val_DF` — fold khó
nhất — cho thứ tự ngược thì kết luận là do fold chứ không do tầm nhìn.

## Mốc để đặt cạnh

| | tham số | tầm nhìn | cv_score |
|---|---:|---:|---:|
| DS-TCN-192 k3n4 no_norm do0.2 | 307.801 | **61** | 0,762714 *(1 seed)* |
| DS-TCN-64 k3n4 no_norm do0.2 | 37.081 | **61** | 0,760878 ± 0,003095 *(3 seed)* |
| LSTM-352 | 1.502.713 | — | 0,756992 ± 0,004156 |

## 1. Chuẩn bị Colab

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Tải mã nguồn.

In [ ]:
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

Lấy `by_user/` và `windows/` từ Drive.

In [ ]:
!python scripts/restore_processed_data_on_drive.py

Khôi phục kết quả `tn2_rf` đã có.

**Bước này quyết định notebook chạy 1,8 giờ hay 2,4 giờ.** Chưa chạy vòng sàng
lọc thì nó in `fold đã có: 0` và làm đủ mười hai fold — vẫn đúng, chỉ lâu hơn.

In [ ]:
# Khôi phục kết quả đã chạy trước khi phiên bị ngắt.
#
# Mỗi tệp nén chứa một bản runs/<thực nghiệm>/summary.csv của riêng nó. Giải
# hết vào cùng một thư mục runs/ thì tệp giải sau ĐÈ summary.csv của tệp trước.
# Mà sorted() xếp "..._a0_corr0.9..." đứng SAU "..._a0.9_corr0.9...", vì trong
# bảng mã ký tự "_" lớn hơn "." — nên bản ít dòng nhất lại là bản đè cuối cùng.
# Sửa: mỗi tệp nén giải vào một thư mục tạm riêng, gộp mọi dòng lại rồi mới ghi
# runs/summary.csv một lần. Thứ tự giải nén không còn ảnh hưởng gì nữa.
import csv, glob, os, shutil, subprocess, tempfile

MAU_ZIP = "/content/drive/MyDrive/mobivital/tn2_rf_*c192*.zip"

rows, seen = [], set()

def collect(summary_path):
    for r in csv.DictReader(open(summary_path)):
        key = (r.get("experiment"), r.get("run_id"))
        if key not in seen:
            seen.add(key)
            rows.append(r)

for f in sorted(glob.glob(MAU_ZIP)):
    tmp = tempfile.mkdtemp()
    subprocess.run(["unzip", "-oq", f, "-d", tmp], check=True)
    for s in glob.glob(tmp + "/*/summary.csv"):
        collect(s)
        os.remove(s)   # gộp xong thì bỏ, để bước chép dưới không đè nhau nữa
    # Checkpoint và curve.csv nằm trong thư mục riêng của từng lần chạy, tên
    # không trùng nhau, nên chép chồng lên runs/ là an toàn.
    for d in os.listdir(tmp):
        shutil.copytree(tmp + "/" + d, "runs/" + d, dirs_exist_ok=True)
    shutil.rmtree(tmp)

# Dòng đã sinh ra trong chính phiên này cũng phải giữ lại.
if os.path.exists("runs/summary.csv"):
    collect("runs/summary.csv")

if rows:
    # run_cv.py tra runs/summary.csv, còn tệp nén chỉ có runs/<thực nghiệm>/summary.csv
    cols = []
    for r in rows:
        for k in r:
            if k not in cols:
                cols.append(k)
    with open("runs/summary.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols, restval="")
        w.writeheader()
        w.writerows(rows)
print("khôi phục", len(rows), "dòng vào runs/summary.csv")

## 2. Kiểm ba bản cài đặt

Số tham số phải ra đúng 310.873 / 313.945 / 317.017.

**Đọc dòng cuối mỗi lệnh.** Phải là `TẤT CẢ ĐẠT`.

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 7 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 9 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

## 3. Chạy đủ 4 fold, một seed

Không có `--folds` nên chạy đủ bốn. Fold nào đã có sẽ in
`đã có kết quả 0.xxxx — bỏ qua, không train lại`.

**kernel 5 — tầm nhìn 121, 310.873 tham số**

In [ ]:
!python scripts/run_cv.py --experiment tn2_rf --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element --seed 0

**kernel 7 — tầm nhìn 181, 313.945 tham số**

In [ ]:
!python scripts/run_cv.py --experiment tn2_rf --model ds_tcn --channels 192 \
    --kernel_size 7 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element --seed 0

**kernel 9 — tầm nhìn 241, 317.017 tham số**

In [ ]:
!python scripts/run_cv.py --experiment tn2_rf --model ds_tcn --channels 192 \
    --kernel_size 9 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element --seed 0

## 4. Cất kết quả

In [ ]:
!python scripts/save_results.py tn2_rf --out tn2_rf_c192_4fold

## 5. Bảng so

Đủ bốn fold nên dòng `TONG` có, `compare_cv` hiện `cv_score` thật. Bảng này gồm
cả các cấu hình 64 kênh nếu chúng cũng nằm trong `runs/tn2_rf/`.

In [ ]:
!python scripts/compare_cv.py --experiment tn2_rf

## 6. Xu hướng có lặp lại không

In điểm từng fold. Hai câu cần trả lời:

1. Ở 192 kênh, thứ tự có giống 64 kênh không — tầm nhìn ngắn hơn thì điểm cao hơn?
2. Thứ tự đó có giữ nguyên ở cả bốn fold, hay chỉ đúng ở `val_KL`?

In [ ]:
import csv, re
RF = {3: 61, 5: 121, 7: 181, 9: 241, 11: 301, 13: 361}
rows = [r for r in csv.DictReader(open("runs/tn2_rf/summary.csv"))
        if "_c192_" in r["run_id"] and r["fold"] != "TONG"]
for r in sorted(rows, key=lambda r: (r["fold"], r["run_id"])):
    k = int(re.search(r"_k(\d+)_", r["run_id"]).group(1))
    print(" ", r["fold"], " kernel", k, " tầm nhìn", RF[k], " ", r["score_macro"])

## 7. Ngắt phiên

In [ ]:
from google.colab import runtime
runtime.unassign()